# Anomaly Computations

In [1]:
import xarray as xr
import pandas as pd
import numpy as np

In [12]:
monthly_averages = xr.open_mfdataset('/pscratch/sd/j/jbbutler/merra2_monthly_data/*.nc4')

In [13]:
monthly_averages_over_years = monthly_averages.groupby(monthly_averages.time.dt.month).mean().compute()

In [16]:
monthly_averages_over_years

<xarray.Dataset> Size: 8MB
Dimensions:  (month: 12, lat: 95, lon: 576)
Coordinates:
  * lat      (lat) float64 760B -86.0 -85.5 -85.0 -84.5 ... -40.0 -39.5 -39.0
  * lon      (lon) float64 5kB -180.0 -179.4 -178.8 -178.1 ... 178.1 178.8 179.4
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
Data variables:
    T2M      (month, lat, lon) float32 3MB 249.1 249.1 249.1 ... 290.3 290.2
    SLP      (month, lat, lon) float32 3MB 9.922e+04 9.921e+04 ... 1.014e+05
    TQV      (month, lat, lon) float32 3MB 1.144 1.143 1.143 ... 22.72 22.42
Attributes: (12/32)
    History:                           Original file generated: Thu May  7 21...
    Filename:                          MERRA2_100.instM_2d_asm_Nx.198001.nc4
    Comment:                           GMAO filename: d5124_m2_jan79.inst1_2d...
    Conventions:                       CF-1
    Institution:                       NASA Global Modeling and Assimilation ...
    References:                        http://gmao.gsfc.nasa.gov
    ...                                ...
    Contact:                           http://gmao.gsfc.nasa.gov
    identifier_product_doi:            10.5067/5ESKGQTZG7FO
    RangeBeginningTime:                00:00:00.000000
    RangeEndingTime:                   23:00:00.000000
    DODS_EXTRA.Unlimited_Dimension:    time
    history:                           2025-06-04 00:42:26 GMT Hyrax-1.16.3 h...

In [22]:
time1 = xr.open_dataset('/pscratch/sd/j/jbbutler/merra2_data_T2m_V10m_SLP_IWV/inst1_2d_asm_Nx.20221227.nc4.nc4')
time2 = xr.open_dataset('/pscratch/sd/j/jbbutler/merra2_data_T2m_V10m_SLP_IWV/inst1_2d_asm_Nx.20221228.nc4.nc4')

full_da = xr.concat([time1, time2], dim='time')
actual_da = full_da['SLP']

In [27]:
climatology = monthly_averages_over_years['SLP']
climatology = climatology.assign_coords(lat=climatology.lat.round(5), lon=climatology.lon.round(5))
actual_da = actual_da.assign_coords(lat=actual_da.lat.round(5), lon=actual_da.lon.round(5))

In [28]:
single_var_da = xr.apply_ufunc(lambda da, clim: da-clim, actual_da.groupby('time.month'), climatology).drop_vars('month')

In [38]:
single_var_da

<xarray.DataArray 'SLP' (time: 48, lat: 95, lon: 576)> Size: 11MB
array([[[-820.47656, -812.65625, -805.5547 , ..., -846.97656,
         -837.6328 , -828.96094],
        [-709.60156, -698.9297 , -690.22656, ..., -739.71875,
         -729.72656, -719.53125],
        [-681.5156 , -669.46094, -655.91406, ..., -706.7656 ,
         -699.1094 , -691.7969 ],
        ...,
        [ 675.39844,  657.7344 ,  647.75   , ...,  763.53906,
          711.6094 ,  688.59375],
        [ 635.15625,  615.09375,  597.91406, ...,  743.39844,
          676.4922 ,  650.7969 ],
        [ 594.1172 ,  568.0078 ,  542.0156 , ...,  707.25   ,
          653.7969 ,  613.25   ]],

       [[-775.0469 , -765.72656, -756.625  , ..., -806.0469 ,
         -795.2031 , -785.03125],
        [-676.6719 , -665.     , -655.7969 , ..., -708.78906,
         -698.2969 , -687.60156],
        [-656.58594, -644.03125, -630.4844 , ..., -680.83594,
         -673.6797 , -666.3672 ],
...
        [1090.5156 , 1086.8516 , 1079.8672 , ..., 1061.6562 ,
         1088.7266 , 1090.7109 ],
        [1066.2734 , 1053.2109 , 1044.0312 , ..., 1070.5156 ,
         1083.6094 , 1079.9141 ],
        [1039.2344 , 1019.125  , 1008.1328 , ..., 1054.3672 ,
         1084.9141 , 1063.3672 ]],

       [[-563.59375, -561.27344, -559.0469 , ..., -571.71875,
         -568.875  , -566.2031 ],
        [-560.21875, -559.1719 , -559.21875, ..., -560.71094,
         -560.96875, -560.77344],
        [-640.1328 , -631.5781 , -622.53125, ..., -635.5078 ,
         -640.10156, -644.91406],
        ...,
        [1090.4062 , 1086.7422 , 1057.7578 , ..., 1052.5469 ,
         1080.6172 , 1089.6016 ],
        [1065.1641 , 1057.1016 , 1021.9219 , ..., 1054.4062 ,
         1073.5    , 1074.8047 ],
        [1036.125  , 1023.0156 ,  988.02344, ..., 1026.2578 ,
         1075.8047 , 1055.2578 ]]], dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 384B 2022-12-27 ... 2022-12-28T23:00:00
  * lat      (lat) float64 760B -86.0 -85.5 -85.0 -84.5 ... -40.0 -39.5 -39.0
  * lon      (lon) float64 5kB -180.0 -179.4 -178.8 -178.1 ... 178.1 178.8 179.4